# Part 2A-xii: ML Training with Weights & Biases (W&B)

**Objective:** Integrate W&B for experiment tracking, logging, and visualization.

---

In [1]:
!pip install wandb -q
import wandb
import numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import torch, torch.nn as nn

# Initialize W&B (use anonymous mode for demo)
wandb.login(anonymous="allow")
print("W&B initialized!")

(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
X_train = X_train.astype("float32").reshape(-1, 784) / 255.0
X_test = X_test.astype("float32").reshape(-1, 784) / 255.0

wandb: WARNING The anonymous parameter to wandb.login() has no effect and will be removed in future versions.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.
wandb: Paste your API key and hit enter:

 ··········


wandb: ERROR Invalid API key: API key must have 40+ characters, has 1.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


W&B initialized!
29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


## TensorFlow with W&B
W&B provides a Keras callback that automatically logs metrics, model architecture, and more.

In [2]:
# Experiment 1: TF with W&B Callback
from wandb.integration.keras import WandbMetricsLogger

config = {
    "learning_rate": 1e-3,
    "epochs": 5,
    "batch_size": 256,
    "architecture": "MLP",
    "dataset": "Fashion-MNIST",
    "hidden_units": [256, 128],
    "dropout": 0.3,
    "optimizer": "adam"
}

run = wandb.init(project="cmpe258-demo", config=config, name="tf_fashion_mnist")

model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(config["hidden_units"][0], activation='relu'),
    layers.Dropout(config["dropout"]),
    layers.Dense(config["hidden_units"][1], activation='relu'),
    layers.Dropout(config["dropout"]),
    layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=config["learning_rate"]),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    epochs=config["epochs"],
    batch_size=config["batch_size"],
    validation_split=0.2,
    callbacks=[WandbMetricsLogger()],
    verbose=1
)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
wandb.log({"test_accuracy": test_acc, "test_loss": test_loss})
print(f"Test Accuracy: {test_acc:.4f}")

# Log confusion matrix
from sklearn.metrics import confusion_matrix
y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
wandb.log({"confusion_matrix": wandb.plot.confusion_matrix(
    y_true=y_test, preds=y_pred,
    class_names=['T-shirt','Trouser','Pullover','Dress','Coat','Sandal','Shirt','Sneaker','Bag','Boot']
)})

wandb.finish()

Epoch 1/5
188/188 ━━━━━━━━━━━━━━━━━━━━ 7s 20ms/step - accuracy: 0.7463 - loss: 0.7240 - val_accuracy: 0.8354 - val_loss: 0.4587
Epoch 2/5
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8339 - loss: 0.4645 - val_accuracy: 0.8610 - val_loss: 0.3880
Epoch 3/5
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8508 - loss: 0.4154 - val_accuracy: 0.8690 - val_loss: 0.3661
Epoch 4/5
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8616 - loss: 0.3880 - val_accuracy: 0.8690 - val_loss: 0.3589
Epoch 5/5
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8682 - loss: 0.3657 - val_accuracy: 0.8747 - val_loss: 0.3426
Test Accuracy: 0.8683


epoch/accuracy,▁▆▇██
epoch/epoch,▁▃▅▆█
epoch/learning_rate,▁▁▁▁▁
epoch/loss,█▃▂▁▁
epoch/val_accuracy,▁▆▇▇█
epoch/val_loss,█▄▂▂▁
test_accuracy,▁
test_loss,▁
epoch/accuracy,0.86823
epoch/epoch,4
epoch/learning_rate,0.001


## PyTorch with W&B — Manual Logging

In [3]:
# Experiment 2: PyTorch with manual W&B logging
config_pt = {
    "learning_rate": 1e-3,
    "epochs": 5,
    "batch_size": 256,
    "architecture": "MLP-PyTorch",
    "hidden_units": [256, 128],
    "dropout": 0.3,
}

run = wandb.init(project="cmpe258-demo", config=config_pt, name="pt_fashion_mnist")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class WBNet(nn.Module):
    def __init__(self, h1, h2, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, h1), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h1, h2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h2, 10))
    def forward(self, x): return self.net(x)

pt_model = WBNet(config_pt["hidden_units"][0], config_pt["hidden_units"][1], config_pt["dropout"]).to(device)
opt = torch.optim.Adam(pt_model.parameters(), lr=config_pt["learning_rate"])
crit = nn.CrossEntropyLoss()

wandb.watch(pt_model, crit, log="all", log_freq=100)

train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train)),
    batch_size=config_pt["batch_size"], shuffle=True)
test_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test)),
    batch_size=config_pt["batch_size"])

for epoch in range(config_pt["epochs"]):
    pt_model.train()
    running_loss = correct = total = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        out = pt_model(xb)
        loss = crit(out, yb)
        loss.backward(); opt.step()
        running_loss += loss.item() * xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        total += yb.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    pt_model.eval()
    val_loss = val_correct = val_total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = pt_model(xb)
            val_loss += crit(out, yb).item() * xb.size(0)
            val_correct += (out.argmax(1) == yb).sum().item()
            val_total += yb.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss, "train_accuracy": train_acc,
        "val_loss": val_loss, "val_accuracy": val_acc,
        "learning_rate": opt.param_groups[0]['lr'],
    })
    print(f"Epoch {epoch+1}/{config_pt['epochs']} — train_acc: {train_acc:.4f}, val_acc: {val_acc:.4f}")

wandb.log({"final_test_accuracy": val_acc})
wandb.finish()
print(f"\nFinal Test Accuracy: {val_acc:.4f}")
print("Check your W&B dashboard for full experiment tracking!")

/tmp/ipykernel_4947/4104758041.py:31: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  torch.utils.data.TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train)),


Epoch 1/5 — train_acc: 0.7452, val_acc: 0.8353
Epoch 2/5 — train_acc: 0.8374, val_acc: 0.8468
Epoch 3/5 — train_acc: 0.8567, val_acc: 0.8563
Epoch 4/5 — train_acc: 0.8639, val_acc: 0.8670
Epoch 5/5 — train_acc: 0.8698, val_acc: 0.8618


epoch,▁▃▅▆█
final_test_accuracy,▁
learning_rate,▁▁▁▁▁
train_accuracy,▁▆▇██
train_loss,█▃▂▁▁
val_accuracy,▁▄▆█▇
val_loss,█▅▃▁▁
epoch,5
final_test_accuracy,0.8618
learning_rate,0.001
train_accuracy,0.86978



Final Test Accuracy: 0.8618
Check your W&B dashboard for full experiment tracking!


## Key Takeaways
- W&B provides: experiment tracking, hyperparameter sweeps, model comparison, artifact management
- **TF**: use `WandbMetricsLogger` callback for automatic logging
- **PyTorch**: use `wandb.log()` for manual logging, `wandb.watch()` for gradient tracking
- **Sweeps**: automate hyperparameter search with random/Bayesian methods
- All experiments are logged to a web dashboard for easy comparison
- Free tier is sufficient for academic use